In [ ]:
import pandas as pd
import numpy as np 

In [ ]:
timestamp = '04-22_02-08-47'
part = '7'
ws = f'/data/geraugi/plural/dataset_files/xplainer_inference_2025-{timestamp}'
confusion_path = f'{ws}/confusion_matrix_part{part}.csv'
descriptor_probs_path = f'{ws}/descriptor_probs_long_part{part}.csv'
probs_path = f'{ws}/xp_inf_probabilities_{part}.csv'
neg_probs_path = f'{ws}/xp_inf_neg_probabilities_{part}.csv'
gt_path = f'{ws}/xp_inf_labels_{part}.csv'
preds_path = f'{ws}/xp_inf_predictions_{part}.csv'

confusion_matrix_df = pd.read_csv(confusion_path)
desc_probs_df = pd.read_csv(descriptor_probs_path)
dis_probs_df = pd.read_csv(probs_path)
dis_neg_probs_df = pd.read_csv(neg_probs_path)
gt_df = pd.read_csv(gt_path)
preds_df = pd.read_csv(preds_path)


In [ ]:
desc_probs_df.head(5)

In [ ]:
confusion_matrix_df = confusion_matrix_df.rename(columns={'Unnamed: 0': 'dicom_id'}) \
                                         .set_index('dicom_id')
dis_probs_df        = dis_probs_df       .rename(columns={'Unnamed: 0': 'dicom_id'}) \
                                         .set_index('dicom_id')
dis_neg_probs_df    = dis_neg_probs_df   .rename(columns={'Unnamed: 0': 'dicom_id'}) \
                                         .set_index('dicom_id')

In [ ]:
from IPython.display import display
conf_long = (
    confusion_matrix_df
      .reset_index()                       # bring dicom_id back as a column if it was the index
      .melt(
         id_vars='dicom_id',              # keep dicom_id
         var_name='disease',              # new column for what used to be each disease column name
         value_name='res'                 # new column for TP/TN/FP/FN values
      )
)

# — 2) Parse disease out of each descriptor
# desc_probs_df has index=dicom_id, columns: descriptor, prob, neg_prob
# and descriptor strings end in "indicating <disease>"
desc = desc_probs_df.copy()
desc['disease'] = desc['descriptor'].str.extract(r'indicating\s+(.+)$')[0].str.strip()


# — 3) Merge and aggregate
merged = (
    desc
      .reset_index()        # make dicom_id a column
      .merge(
         conf_long,
         on=['dicom_id', 'disease'],
         how='inner'
      )
)

merged['dif_prob'] = merged['prob'] - merged['neg_prob']
# Now each row has: dicom_id, descriptor, prob, neg_prob, disease, res

# Group by disease / descriptor / res and compute:
stats = (
    merged
      .groupby(['disease','descriptor','res'])
      .agg(
         #n               = ('res',      'count'),
         mean_prob       = ('prob',     'mean'),
         std_prob        = ('prob',     'std'),
         mean_neg_prob   = ('neg_prob', 'mean'),
         std_neg_prob    = ('neg_prob', 'std'),
         mean_dif_prob   = ('dif_prob', 'mean'),
         std_dif_prob    = ('dif_prob', 'std'),
      )
      .reset_index()
)
#stats_fp = stats.query("res == 'FP'")
merged_fp = merged.query("res == 'FP'")
#display(stats_fp)
display(merged_fp)

In [ ]:
# 1) filter FP & dif_prob < 0
fp_neg = merged[(merged['res'] == 'FP') & (merged['dif_prob'] < 0)]

# 2) count how many times each descriptor appears
desc_counts = (
    fp_neg
      .groupby('descriptor')
      .size()
      .reset_index(name='count')
      .sort_values('count', ascending=False)
)

print(desc_counts)

In [ ]:
# 1) filter FP & dif_prob < 0
tp_neg = merged[(merged['res'] == 'TP') & (merged['dif_prob'] < 0)]

# 2) count how many times each descriptor appears
desc_counts = (
    tp_neg
      .groupby('descriptor')
      .size()
      .reset_index(name='count')
      .sort_values('count', ascending=False)
)

print(desc_counts)

In [ ]:
# 1. Suffix the columns
res_df      = confusion_matrix_df.add_suffix('_res')
prob_df     = dis_probs_df.add_suffix('_prob')
neg_prob_df = dis_neg_probs_df.add_suffix('_neg_prob')

# 2. Concatenate on the index
merged_df = pd.concat([res_df, prob_df, neg_prob_df], axis=1, join='inner')

merged_df.head(5)

In [ ]:
diseases = [
            "Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Emphysema",
            "Fracture", "Hernia", "Lung Opacity", "Pleural Effusion", "Pleural Thickening",
            "Pneumonia", "Pneumothorax", "No Findings"
        ]
for disease in diseases:
    prob_col = f"{disease}_prob"
    neg_col  = f"{disease}_neg_prob"
    diff_col = f"{disease}_dif_prob"
    
    # only if both source columns exist
    if prob_col in merged_df.columns and neg_col in merged_df.columns:
        merged_df[diff_col] = merged_df[prob_col] - merged_df[neg_col]

In [ ]:

for disease in diseases:
    res_col      = f"{disease}_res"
    prob_col     = f"{disease}_prob"
    neg_prob_col = f"{disease}_neg_prob"
    dif_prob_col = f"{disease}_dif_prob"
    
    # group by TP/TN/FP/FN
    grp = merged_df.groupby(res_col)
    grp.head()
    # aggregate counts + stats for both prob and neg_prob
    agg = grp[res_col, prob_col, neg_prob_col, dif_prob_col].agg(
        count       =  (res_col, 'count'),      # count per group
        mean_prob   =  (prob_col, 'mean'),
        std_prob    =  (prob_col, 'std'),
        mean_neg    =  (neg_prob_col, 'mean'),
        std_neg     =  (neg_prob_col, 'std'),
        mean_dif    =  (dif_prob_col, 'mean'),
        std_dif     =  (dif_prob_col, 'std')
    )
    
    # flatten MultiIndex columns
    agg.columns = ['n', 'mean_prob', 'std_prob', 'mean_neg_prob', 'std_neg_prob','mean_dif_prob', 'std_dif_prob'] 
    
    agg = agg.reset_index()  # bring res (TP/TN/…) back as a column
    print(agg)
